# Linear regression, end to end

The four lessons take the method apart. This notebook puts it back together as the workflow you would actually ship: real data, a leak-free pipeline, a tuned penalty, a score from data the search never touched, and a saved artifact.

Order is the whole point. The split happens before any scaling, and the scaler lives *inside* the pipeline so it refits on each cross-validation fold. Scaling the full dataset first is the most common way to leak information from held-out rows into training, and it buys a score that does not survive contact with new data.

`fetch_california_housing` downloads the dataset on first run and caches it afterwards, so the first cell needs a network connection.

In [1]:
import joblib
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Load data & split (Never scale before this step!)
housing = fetch_california_housing(as_frame=True)
X = housing.data[['MedInc', 'AveRooms', 'AveOccup', 'HouseAge']]
y = housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Create a Pipeline (Combines scaling and modeling safely)
# This ensures the scaler only learns parameters from training folds during CV.
pipeline = Pipeline(
    [
        ('scaler', StandardScaler()),
        (
            'model',
            Ridge(),
        ),  # Change to Lasso() if you need feature elimination
    ]
)

# 3. Setup Hyperparameter Tuning with Cross-Validation
# We test different 'alpha' values to find which regularization strength works best.
param_grid = {'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,  # 5-fold cross-validation
    scoring='neg_mean_squared_error',
    n_jobs=-1,
)

# 4. Train and optimize
grid_search.fit(X_train, y_train)

print(f'Best Hyperparameters: {grid_search.best_params_}')
print(f'Best Cross-Validation MSE: {-grid_search.best_score_:.4f}')

# 5. Evaluate on the unseen test set
best_model = grid_search.best_estimator_
test_score = best_model.score(X_test, y_test)
print(f'Test R^2 Score: {test_score:.4f}')

# 6. Save the production artifact for deployment
joblib.dump(best_model, 'ridge_housing_pipeline.pkl')
print('Model artifact saved successfully!')

Best Hyperparameters: {'model__alpha': 100.0}
Best Cross-Validation MSE: 0.6482
Test R^2 Score: 0.4984
Model artifact saved successfully!

## What the output is telling you

- **The winning alpha is 100.0 — the largest value in the grid.** When the best value sits at the edge, the grid is too narrow to have found an optimum. Widen it (try up to 1000) and refit before believing the number.
- **Test R² of 0.4984** means four features explain about half the variation in house value. That is honest rather than good: it is a baseline for anything more elaborate to beat, not a finished model.
- **MSE and R² answer different questions.** The search optimised mean squared error; the test line reports R². Tuning on one and reporting the other is fine, as long as you say which is which.

## Extend this notebook

- Add the remaining columns from `housing.data` and see how much the test score moves.
- Swap `Ridge()` for `Lasso()` in the pipeline and check which features survive the penalty.
- Plot residuals against predictions, looking for curvature or spread that changes across the range.
- Pin your library versions next to the `.pkl`. A pickled pipeline is tied to the scikit-learn that wrote it, and will not reliably load into a different one.